<a href="https://colab.research.google.com/github/vaidegiarch/aerial-obj-classification/blob/main/aerial_obj_streamlit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
!pip install -q streamlit
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
import subprocess
subprocess.Popen(["./cloudflared-linux-amd64", "tunnel", "--url", "http://localhost:8501"])
!nohup /content/cloudflared-linux-amd64 tunnel --url http://localhost:8501 &

--2026-04-25 14:53:28--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64 [following]
--2026-04-25 14:53:28--  https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/731ab2f8-6b77-4adb-a7b3-1104525e9d72?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-04-25T15%3A33%3A30Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-04-25T1

In [22]:
from tensorflow.keras.models import load_model

model = load_model("best_cnn_model.keras")

In [23]:
%%writefile app.py

import streamlit as st
import numpy as np
from tensorflow.keras.models import load_model
from PIL import Image

# Load trained model
model = load_model("best_cnn_model.keras")

# Title
st.title("🦅 Bird vs 🚁 Drone Classifier")
st.write("Upload an image to classify whether it's a Bird or a Drone")

# ---- Prediction Function with TTA ----
def predict_tta(model, img_array):
    preds = []

    # Original
    preds.append(model.predict(img_array, verbose=0)[0][0])

    # Horizontal flip
    flipped = np.flip(img_array, axis=2)
    preds.append(model.predict(flipped, verbose=0)[0][0])

    # Rotate 90°
    rotated1 = np.rot90(img_array, k=1, axes=(1, 2))
    preds.append(model.predict(rotated1, verbose=0)[0][0])

    # Rotate 180°
    rotated2 = np.rot90(img_array, k=2, axes=(1, 2))
    preds.append(model.predict(rotated2, verbose=0)[0][0])

    return np.mean(preds)

# Upload file
uploaded_file = st.file_uploader("Choose an image...", type=["jpg", "jpeg", "png"])

if uploaded_file is not None:

    # Display uploaded image
    image = Image.open(uploaded_file).convert("RGB")
    st.image(image, caption="Uploaded Image", width="stretch")

    # ---- Preprocess ----
    img = image.resize((224, 224))
    img = np.array(img) / 255.0
    img = np.expand_dims(img, axis=0)

    # ---- Predict ----
    prediction = predict_tta(model, img)

    # ---- Display raw prediction ----
    st.write(f"Raw prediction score: {prediction:.3f}")

    # ---- Classification Logic ----
    if prediction > 0.75:
        label = "🚁 Drone"
        confidence = prediction
    elif prediction < 0.30:
        label = "🦅 Bird"
        confidence = 1 - prediction
    else:
        label = "⚠️ Not Sure"
        confidence = prediction

    # ---- Output ----
    st.subheader(f"Prediction: {label}")
    st.write(f"Confidence: {confidence:.2f}")

Overwriting app.py


In [24]:
!streamlit run /content/app.py &>/content/logs.txt &

In [25]:
!grep -o 'https://.*\.trycloudflare.com' nohup.out | head -n 1 | xargs -I {} echo "Your tunnel url {}"

Your tunnel url https://watching-drain-sculpture-pda.trycloudflare.com
